# baseline v3

이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.

Colab의 GPU 환경에서 개발되었습니다.
- 런타임 - 런타임 유형 변경 - GPU로 변경(T4 GPU 등)



# 환경 준비

개발 환경에 필요한 라이브러리 버전을 고정하고 최신 버전으로 라이브러리를 업데이트합니다.

- 아래 셀 실행
- 실행 완료 후 런타임 - 세션 다시 시작

In [5]:
!pip uninstall -y transformers
!pip install -q git+https://github.com/huggingface/transformers accelerate peft bitsandbytes qwen-vl-utils scikit-learn

Found existing installation: transformers 5.5.0.dev0
Uninstalling transformers-5.5.0.dev0:
  Successfully uninstalled transformers-5.5.0.dev0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 79.0 MB/s eta 0:00:00


In [1]:
!pip -q install git+https://github.com/huggingface/transformers accelerate

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
!pip -q install "peft>=0.13.2" "bitsandbytes==0.46.1" datasets pillow pandas --upgrade

In [1]:
!pip -q install git+https://github.com/huggingface/transformers accelerate
!pip -q install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
!pip -q install "peft>=0.13.2" "bitsandbytes==0.46.1" datasets pillow pandas --upgrade

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
import torch
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())

Torch version: 2.10.0+cu128
CUDA version: 12.8
cuDNN version: 91002


# 데이터 준비

개발에 필요한 데이터를 준비합니다.

- train.csv, train 폴더
- test.csv, test 폴더
- sample_submission.csv

본 베이스라인은 colab에서 구글 드라이브를 마운트하여 사용합니다.

데이터를 압축 해제하는데 몇 분 정도의 시간이 소요됩니다.

#### 실습 참고 내용

    챕터 2-2 합성 데이터 실습
    - 구글 드라이브 마운트 : drive()

In [1]:
# 구글드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# 압축 해제ㅛ
!unzip "/content/drive/My Drive/SSFAY_AI.zip" -d "/content/"

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
  inflating: /content/train/train_0074.jpg  
  inflating: /content/train/train_0075.jpg  
  inflating: /content/train/train_0076.jpg  
  inflating: /content/train/train_0077.jpg  
  inflating: /content/train/train_0078.jpg  
  inflating: /content/train/train_0079.jpg  
  inflating: /content/train/train_0080.jpg  
  inflating: /content/train/train_0081.jpg  
  inflating: /content/train/train_0082.jpg  
  inflating: /content/train/train_0083.jpg  
  inflating: /content/train/train_0084.jpg  
  inflating: /content/train/train_0085.jpg  
  inflating: /content/train/train_0086.jpg  
  inflating: /content/train/train_0087.jpg  
  inflating: /content/train/train_0088.jpg  
  inflating: /content/train/train_0089.jpg  
  inflating: /content/train/train_0090.jpg  
  inflating: /content/train/train_0091.jpg  
  inflating: /content/train/train_0092.jpg  
  inflating: /content/train/train_0093.jpg  
  inflating: /content/train/train_0094.jpg  
  inflating: /conte

# 라이브러리, 데이터, 설정

In [3]:
# import os, re, math, random
# import pandas as pd
# from PIL import Image, ImageFile
# from torch.utils.data import Dataset, DataLoader
# from dataclasses import dataclass
# import torch
# from typing import Dict, List, Any
# from transformers import (
#     AutoModelForVision2Seq,
#     AutoProcessor,
#     BitsAndBytesConfig,
#     get_cosine_schedule_with_warmup
# )
# from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
# from tqdm import tqdm


# # 이미지 로드 시 픽셀 제한 해제
# Image.MAX_IMAGE_PIXELS = None
# ImageFile.LOAD_TRUNCATED_IMAGES = True

# # 디바이스 GPU 우선 사용 설정
# device = "cuda" if torch.cuda.is_available() else "cpu"
# USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
# AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
# print("Device:", device)
# print("AMP dtype:", AMP_DTYPE)

# # 사전 학습 모델 정의
# MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
# IMAGE_SIZE = 320
# MAX_NEW_TOKENS = 2
# SEED = 42

# random.seed(SEED)
# torch.manual_seed(SEED)
# if torch.cuda.is_available():
#     torch.cuda.manual_seed_all(SEED)

# # Colab 압축 해제 후 기본 작업 폴더는 /content 이므로 그대로 사용
# train_df = pd.read_csv("/content/train.csv")
# test_df  = pd.read_csv("/content/test.csv")

# # 전체 train set 사용
# train_df["path"] = train_df["ID"].apply(lambda x: f"/content/train/{x}.jpg")
# test_df["path"]  = test_df["ID"].apply(lambda x: f"/content/test/{x}.jpg")


ImportError: cannot import name 'AutoModelForVision2Seq' from 'transformers' (/usr/local/lib/python3.12/dist-packages/transformers/__init__.py)

In [ ]:
import os, re, math, random
import pandas as pd
from PIL import Image, ImageFile
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any, Optional
from sklearn.model_selection import train_test_split
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm import tqdm

# 이미지 로드 설정
Image.MAX_IMAGE_PIXELS = None
ImageFile.LOAD_TRUNCATED_IMAGES = True

# 디바이스 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

print("Device:", device)
print("AMP dtype:", AMP_DTYPE)

# 기본 설정
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
IMAGE_SIZE = 320
MAX_NEW_TOKENS = 2
SEED = 42

# 학습 관련 기본 하이퍼파라미터
EPOCHS = 5
LR = 5e-5
WEIGHT_DECAY = 0.01
GRAD_ACCUM = 4
EARLY_STOPPING_PATIENCE = 2
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
LABEL_SMOOTHING = 0.03

# seed 고정
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 경로 설정
TRAIN_CSV_PATH = "/content/train.csv"
TEST_CSV_PATH = "/content/test.csv"
DEV_CSV_PATH = "/content/dev.csv"

TRAIN_IMAGE_DIR = "/content/train"
TEST_IMAGE_DIR = "/content/test"
DEV_IMAGE_DIR = "/content/dev"

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).replace("\ufeff", "").replace("﻿", "").strip() for c in df.columns]
    return df

def find_column(df: pd.DataFrame, candidates: List[str], name: str, required: bool = True) -> Optional[str]:
    cols = list(df.columns)
    lower_map = {str(c).strip().lower(): c for c in cols}

    for c in candidates:
        if c in cols:
            return c

    for c in candidates:
        key = str(c).strip().lower()
        if key in lower_map:
            return lower_map[key]

    if required:
        raise ValueError(f"{name} 컬럼을 찾지 못했습니다. 현재 컬럼: {cols}")
    return None

def find_id_column(df: pd.DataFrame, df_name: str) -> str:
    return find_column(
        df,
        ["ID", "id", "Id", "image_id", "img_id", "file_id", "filename", "file_name"],
        f"{df_name} ID",
        required=True
    )

def build_image_path(x: Any, image_dir: str) -> str:
    x = str(x).strip()
    if not re.search(r"\.(jpg|jpeg|png|bmp|webp)$", x, flags=re.IGNORECASE):
        x += ".jpg"
    return os.path.join(image_dir, x)

def build_schema(df: pd.DataFrame, df_name: str, require_answer: bool = True) -> Dict[str, Optional[str]]:
    schema = {
        "id": find_id_column(df, df_name),
        "question": find_column(df, ["question", "Question", "prompt", "Prompt"], f"{df_name} question"),
        "a": find_column(df, ["a", "A", "option_a", "Option_A", "choice_a"], f"{df_name} a"),
        "b": find_column(df, ["b", "B", "option_b", "Option_B", "choice_b"], f"{df_name} b"),
        "c": find_column(df, ["c", "C", "option_c", "Option_C", "choice_c"], f"{df_name} c"),
        "d": find_column(df, ["d", "D", "option_d", "Option_D", "choice_d"], f"{df_name} d"),
        "path": "path",
        "answer": find_column(
            df,
            ["answer", "Answer", "label", "Label", "target", "Target"],
            f"{df_name} answer",
            required=require_answer
        )
    }
    return schema

# CSV 로드
train_df = normalize_columns(pd.read_csv(TRAIN_CSV_PATH))
test_df = normalize_columns(pd.read_csv(TEST_CSV_PATH))

dev_df = None
if os.path.exists(DEV_CSV_PATH):
    dev_df = normalize_columns(pd.read_csv(DEV_CSV_PATH))
    print(f"Loaded dev.csv: {len(dev_df)} rows")
else:
    print("dev.csv not found. Will use train split for validation.")

print("train columns:", train_df.columns.tolist())
print("test columns:", test_df.columns.tolist())
if dev_df is not None:
    print("dev columns:", dev_df.columns.tolist())

# ID 컬럼 자동 탐색
train_id_col = find_id_column(train_df, "train_df")
test_id_col = find_id_column(test_df, "test_df")
print("train ID column:", train_id_col)
print("test ID column:", test_id_col)

# 이미지 경로 생성
train_df["path"] = train_df[train_id_col].astype(str).apply(lambda x: build_image_path(x, TRAIN_IMAGE_DIR))
test_df["path"] = test_df[test_id_col].astype(str).apply(lambda x: build_image_path(x, TEST_IMAGE_DIR))

if dev_df is not None:
    try:
        dev_id_col = find_id_column(dev_df, "dev_df")
        dev_df["path"] = dev_df[dev_id_col].astype(str).apply(lambda x: build_image_path(x, DEV_IMAGE_DIR))
        print("dev ID column:", dev_id_col)
    except Exception as e:
        print("dev path 자동 생성 생략:", e)

TRAIN_SCHEMA = build_schema(train_df, "train_df", require_answer=True)
TEST_SCHEMA = build_schema(test_df, "test_df", require_answer=False)
DEV_SCHEMA = None
if dev_df is not None:
    try:
        DEV_SCHEMA = build_schema(dev_df, "dev_df", require_answer=False)
    except Exception as e:
        print("dev schema 자동 생성 생략:", e)

print("TRAIN_SCHEMA:", TRAIN_SCHEMA)
print("TEST_SCHEMA:", TEST_SCHEMA)
if DEV_SCHEMA is not None:
    print("DEV_SCHEMA:", DEV_SCHEMA)

print("train_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)
if dev_df is not None:
    print("dev_df shape:", dev_df.shape)

# 파일 존재 여부 간단 체크
missing_train = (~train_df["path"].apply(os.path.exists)).sum()
missing_test = (~test_df["path"].apply(os.path.exists)).sum()

print(f"Missing train images: {missing_train}")
print(f"Missing test images: {missing_test}")

if dev_df is not None and "path" in dev_df.columns:
    missing_dev = (~dev_df["path"].apply(os.path.exists)).sum()
    print(f"Missing dev images: {missing_dev}")

display(train_df[[train_id_col, "path"]].head())
display(test_df[[test_id_col, "path"]].head())

# 모델, Processor

7.5GB 정도의 모델 다운로드가 진행됩니다. 10~20분 정도가 소요됩니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [6]:
# 양자화
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=AMP_DTYPE,
)

# 프로세서
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE * IMAGE_SIZE,
    max_pixels=IMAGE_SIZE * IMAGE_SIZE,
    trust_remote_code=True,
)

# 사전학습 모델
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=AMP_DTYPE,
    device_map="auto",
    trust_remote_code=True,
)

# 양자화 모델 준비
base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

# LoRA 세팅
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

# PEFT 모델 생성
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

trainable params: 95,178,752 || all params: 8,387,345,408 || trainable%: 1.1348


# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [ ]:
SYSTEM_INSTRUCT = (
    "You are an expert visual material classification system for recyclable waste. "
    "Carefully inspect the image and determine the material type by comparing texture, transparency, gloss, shape, thickness, and packaging cues. "
    "Then compare all answer choices and select the single best answer. "
    "Output exactly one lowercase letter among a, b, c, or d. "
    "Do not output any explanation or extra text."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        "이미지를 자세히 보고 재질과 형태를 판단한 뒤 질문에 답하세요.\n\n"
        f"질문: {question}\n\n"
        "선택지:\n"
        f"a. {a}\n"
        f"b. {b}\n"
        f"c. {c}\n"
        f"d. {d}\n\n"
        "판단할 때 표면 질감, 투명도, 광택, 두께감, 형태, 포장 단서를 함께 보세요.\n"
        "네 선택지를 모두 비교한 뒤 가장 적절한 하나를 고르세요.\n"
        "반드시 a, b, c, d 중 하나만 소문자 한 글자로 출력하세요."
    )

def build_prompt_messages(question, a, b, c, d, image):
    user_text = build_mc_prompt(question, a, b, c, d)
    return [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": user_text}
        ]}
    ]

def build_prompt_messages_from_row(row, schema, image):
    return build_prompt_messages(
        question=str(row[schema["question"]]),
        a=str(row[schema["a"]]),
        b=str(row[schema["b"]]),
        c=str(row[schema["c"]]),
        d=str(row[schema["d"]]),
        image=image
    )

# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [ ]:
# 이전 버전 Dataset 셀입니다. 아래 새 Dataset 셀(다음 셀)만 사용하세요.

In [ ]:
# 커스텀 데이터셋
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, schema, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.schema = schema
        self.train = train

    def __len__(self):
        return len(self.df)

    def _safe_open_image(self, path):
        candidate_paths = [path]
        lower = str(path).lower()
        if lower.endswith(".jpg.jpg"):
            candidate_paths.append(path[:-4])
        elif lower.endswith(".jpeg.jpeg"):
            candidate_paths.append(path[:-5])

        for p in candidate_paths:
            if os.path.exists(p):
                try:
                    return Image.open(p).convert("RGB")
                except Exception as e:
                    print(f"[Image Open Error] {p}: {e}")

        print(f"[Image Error] file not found: {path}")
        return Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), color=(255, 255, 255))

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = self._safe_open_image(row[self.schema["path"]])

        prompt_messages = build_prompt_messages_from_row(row, self.schema, img)

        sample = {
            "image": img,
            "prompt_messages": prompt_messages
        }

        if self.train:
            sample["answer"] = str(row[self.schema["answer"]]).strip().lower()

        return sample

@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        images = [sample["image"] for sample in batch]

        if self.train:
            prompt_texts = []
            full_texts = []

            for sample in batch:
                prompt_messages = sample["prompt_messages"]
                gold = sample["answer"]

                full_messages = prompt_messages + [
                    {"role": "assistant", "content": [{"type": "text", "text": gold}]}
                ]

                prompt_text = self.processor.apply_chat_template(
                    prompt_messages,
                    tokenize=False,
                    add_generation_prompt=True
                )
                full_text = self.processor.apply_chat_template(
                    full_messages,
                    tokenize=False,
                    add_generation_prompt=False
                )

                prompt_texts.append(prompt_text)
                full_texts.append(full_text)

            enc_full = self.processor(
                text=full_texts,
                images=images,
                padding=True,
                return_tensors="pt"
            )

            enc_prompt = self.processor(
                text=prompt_texts,
                images=images,
                padding=True,
                return_tensors="pt"
            )

            labels = enc_full["input_ids"].clone()
            pad_token_id = self.processor.tokenizer.pad_token_id
            if pad_token_id is not None:
                labels[labels == pad_token_id] = -100

            for i in range(labels.size(0)):
                prompt_len = int(enc_prompt["attention_mask"][i].sum().item())
                labels[i, :prompt_len] = -100

            enc_full["labels"] = labels
            return enc_full

        prompt_texts = []
        for sample in batch:
            prompt_text = self.processor.apply_chat_template(
                sample["prompt_messages"],
                tokenize=False,
                add_generation_prompt=True
            )
            prompt_texts.append(prompt_text)

        enc = self.processor(
            text=prompt_texts,
            images=images,
            padding=True,
            return_tensors="pt"
        )
        return enc

# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [ ]:
# 이전 버전 DataLoader 셀입니다. 아래 새 DataLoader 셀(다음 셀)만 사용하세요.

In [ ]:
answer_col = TRAIN_SCHEMA["answer"]

label_series = train_df[answer_col].astype(str).str.strip().str.lower()
use_stratify = label_series.value_counts().min() >= 2

try:
    train_subset, valid_subset = train_test_split(
        train_df,
        test_size=0.1,
        random_state=SEED,
        shuffle=True,
        stratify=label_series if use_stratify else None
    )
    print("Stratified split used." if use_stratify else "Random split used (stratify unavailable).")
except Exception as e:
    print(f"train/valid split에서 stratify 실패, random split으로 대체: {e}")
    train_subset, valid_subset = train_test_split(
        train_df,
        test_size=0.1,
        random_state=SEED,
        shuffle=True,
        stratify=None
    )

train_subset = train_subset.reset_index(drop=True)
valid_subset = valid_subset.reset_index(drop=True)

train_ds = VQAMCDataset(train_subset, processor, TRAIN_SCHEMA, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, TRAIN_SCHEMA, train=True)

train_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=True,
    collate_fn=DataCollator(processor, train=True),
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=1,
    shuffle=False,
    collate_fn=DataCollator(processor, train=True),
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print(f"train_subset: {len(train_subset)}")
print(f"valid_subset: {len(valid_subset)}")
print("train label counts:")
print(train_subset[answer_col].astype(str).str.strip().str.lower().value_counts().sort_index())
print("valid label counts:")
print(valid_subset[answer_col].astype(str).str.strip().str.lower().value_counts().sort_index())

# fine-tuning

- 200개만 학습 : 10~20분 소요

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
# 이전 버전 학습 루프 셀입니다. 아래 새 학습 셀(다음 셀)만 사용하세요.

In [ ]:
from tqdm.auto import tqdm
import os
import torch.nn.functional as F
from contextlib import nullcontext

# 4bit + device_map="auto" 모델은 model.to(device)로 다시 옮기지 않습니다.
MODEL_DEVICE = next(model.parameters()).device
print("Model device:", MODEL_DEVICE)

if hasattr(model, "config"):
    model.config.use_cache = False

trainable_params = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

num_training_steps = EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
num_warmup_steps = int(num_training_steps * 0.03)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(torch.cuda.is_available() and AMP_DTYPE == torch.float16)
)

SAVE_DIR = "/content/qwen2_5_vl_7b_lora"
os.makedirs(SAVE_DIR, exist_ok=True)

best_val_loss = float("inf")
early_stop_counter = 0

def move_batch_to_device(batch, target_device):
    moved = {}
    for k, v in batch.items():
        moved[k] = v.to(target_device) if torch.is_tensor(v) else v
    return moved

def autocast_context():
    if torch.cuda.is_available():
        return torch.amp.autocast(device_type="cuda", dtype=AMP_DTYPE)
    return nullcontext()

def compute_masked_lm_loss(logits, labels, label_smoothing=0.0):
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()

    return F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        ignore_index=-100,
        reduction="mean",
        label_smoothing=label_smoothing
    )

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    running_loss = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")

    for step, batch in enumerate(progress_bar, start=1):
        batch = move_batch_to_device(batch, MODEL_DEVICE)
        labels = batch["labels"]
        model_inputs = {k: v for k, v in batch.items() if k != "labels"}

        with autocast_context():
            outputs = model(**model_inputs)
            loss = compute_masked_lm_loss(
                outputs.logits,
                labels,
                label_smoothing=LABEL_SMOOTHING
            )
            scaled_loss = loss / GRAD_ACCUM

        if scaler.is_enabled():
            scaler.scale(scaled_loss).backward()
        else:
            scaled_loss.backward()

        running_loss += float(loss.item())
        progress_bar.set_postfix({"loss": f"{running_loss / step:.4f}"})

        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            if scaler.is_enabled():
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
                optimizer.step()

            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

    model.eval()
    val_loss_sum = 0.0
    val_steps = 0

    with torch.no_grad():
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1} [valid]", unit="batch"):
            vb = move_batch_to_device(vb, MODEL_DEVICE)
            labels = vb["labels"]
            model_inputs = {k: v for k, v in vb.items() if k != "labels"}

            with autocast_context():
                outputs = model(**model_inputs)
                val_loss = compute_masked_lm_loss(outputs.logits, labels, label_smoothing=0.0)

            val_loss_sum += float(val_loss.item())
            val_steps += 1

    avg_val_loss = val_loss_sum / max(val_steps, 1)
    print(f"[Epoch {epoch+1}] valid loss = {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        early_stop_counter = 0
        model.save_pretrained(SAVE_DIR)
        processor.save_pretrained(SAVE_DIR)
        print(f"Best model saved to {SAVE_DIR}")
    else:
        early_stop_counter += 1
        print(f"No improvement. Early stop counter: {early_stop_counter}/{EARLY_STOPPING_PATIENCE}")

        if early_stop_counter >= EARLY_STOPPING_PATIENCE:
            print("Early stopping triggered.")
            break

print("Training finished. Best valid loss:", best_val_loss)

# inference

30분~1시간 소요

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
# 데이터 파서 : 모델의 응답에서 선지를 추출
def extract_choice(text: str) -> str:
    text = text.strip().lower()

    match = re.findall(r"\b([abcd])\b", text)
    if match:
        return match[-1]

    for ch in text:
        if ch in ["a", "b", "c", "d"]:
            return ch

    return "a"

def safe_open_image(path):
    candidate_paths = [path]
    lower = str(path).lower()
    if lower.endswith(".jpg.jpg"):
        candidate_paths.append(path[:-4])
    elif lower.endswith(".jpeg.jpeg"):
        candidate_paths.append(path[:-5])

    for p in candidate_paths:
        if os.path.exists(p):
            try:
                return Image.open(p).convert("RGB")
            except Exception as e:
                print(f"[Inference Image Open Error] {p}: {e}")

    print(f"[Inference Image Error] file not found: {path}")
    return Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), color=(255, 255, 255))

# 추론을 위해 eval 모드
model.eval()
preds = []

# 추론 루프
for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]
    img = safe_open_image(row[TEST_SCHEMA["path"]])

    user_text = build_mc_prompt(
        row[TEST_SCHEMA["question"]],
        row[TEST_SCHEMA["a"]],
        row[TEST_SCHEMA["b"]],
        row[TEST_SCHEMA["c"]],
        row[TEST_SCHEMA["d"]]
    )

    messages = [
        {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
        {"role":"user","content":[
            {"type":"image","image":img},
            {"type":"text","text":user_text}
        ]}
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=[img],
        return_tensors="pt"
    ).to(MODEL_DEVICE)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            eos_token_id=processor.tokenizer.eos_token_id
        )

    generated_ids = out_ids[:, inputs["input_ids"].shape[1]:]
    output_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

    pred = extract_choice(output_text)
    preds.append(pred)

# 제출 파일 생성
submission = pd.DataFrame({"id": test_df[test_id_col], "answer": preds})
submission.to_csv("/content/submission.csv", index=False)
print("Saved /content/submission.csv")
display(submission.head())

In [ ]:
# 모델 응답 예시 확인용 셀은 제거했습니다. 필요하면 output_text 변수를 직접 출력하세요.